# **Preparation Notebook**



---
## Setup Environment

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT2",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

---
## Student Information

In [ ]:
student_name = "Nonthawat Praisompong"
student_id = "25233750"

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

### 0.b Import Packages

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import altair as alt
import scipy.stats as stats

---
## A. Feature Selection

### In this project, we decided to use 3 approaches for feature selection.


*   Domain knowledge (removing the feature that is not related to credit_rating based on the prior knowledge)
*   ANOVA (Statistic Tesing for numeric columns)
*   Chi-squre (Statistic Tesing for category columns)

However, we have to do ANOVA and Chi-square after splitting the data to prevent data leakage. In summary, we will continue ANOVA and Chi-square after we splitting the data.

## A.0 # Load Data

In [ ]:
folder_path = '/content/gdrive/MyDrive/36106/assignment/AT2/data'

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
try:
  df = pd.read_csv(at.folder_path / "credit_rating.csv")
except Exception as e:
  print(e)

In [ ]:
# making the copy of the dataset for data manipulation
df_selection = df.copy()

### A.1 Approach 1: Domain knowledge and what we have observe from EDA notebook


In [ ]:
# 26 features will be removed
# personal information: Most of these variables are not related to the target variables and add model's noise
# By doing this help to reduce the complexity of the model.
# birth_country, street_suffix and city have too much unique and imbalance which might cause bias on the result.

personal = ['customer_id', 'prefix', 'full_name', 'dob', 'birth_country', 'email', 'phone_number', 'secondary_address',
            'building_number', 'street_name', 'street_suffix', 'city', 'postcode', 'state_abbr']

# unrelated feature (the characteristic of the card does not add any significant value), removing them is the best approach.
unrelated_1 = ['cc_number', 'cc_expiry', 'cc_security_code']

# unrelated 2 (These columns require OHE and it will increase model's complexity)
# Also, these columns are dramatically imbalanced, which is not good for modelling. (90% of the distribution is no_loan, indicating an imbalance)
# Whcih mean removing them is the best strategy
unrelated_2 = ['last_9_loan_type', 'last_8_loan_type', 'last_7_loan_type', 'last_6_loan_type', 'last_5_loan_type', 'last_4_loan_type']

In [ ]:
# Removing the feature which are or related to customer personal data
df_selection = df_selection.drop(columns = personal)
df_selection = df_selection.drop(columns = unrelated_1)
df_selection = df_selection.drop(columns = unrelated_2)

In [ ]:
# <Student to fill this section and then remove this comment>
feature_selection_1_insights = """
According to, we cannot do AVOVA and Chi-square for feature selection because it causes data leakage. Removing variables that do not relate to the credit_rating,
along with the observation result that we got from the EDA notebook. Primarily, I believe that it helps to reduce model complexity and reduce the model overfitting (from data leakage).
On top of that, this method saving time and reduce noise which these features do not app any significant into the model performance. As a result, 26 variables were removed from the model.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_1_insights', value=feature_selection_1_insights)

### A.z Final Selection of Features

In [ ]:
first_selection = df_selection.columns

In [ ]:
# 25 features will be selected for the modelling parts
features_list = list(first_selection)

In [ ]:
feature_selection_explanations = """
This is an initial of feature selection without proving by statistic approach (ANOVA and Chi-squres). However, this approach will help in reducing in model complexity by removing unrelated features from the model.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## B. Data Cleaning

From the EDA result, there is no missing and duplicated values in the dataset.
However, there are 3 issue that I found during the EDA.

* Data type: actually, it is not wrong, but changing the data type helps the model to work more precisely.
* In correct value: some columns contain the missing value, which requires correcting.
* rename a column

However, there are some columns with outliers(numerical) and imbalance (categorical distribution) which we have to deal with it.


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets
try:
  df_clean = df[features_list].copy()
except Exception as e:
  print(e)

In [ ]:
# Rename _monthly_invested_amount
df_clean.rename(columns = {"_monthly_invested_amount" : "monthly_invested_amount"}, inplace = True)


### B.1 Fixing "\<Fixing data type to suitable for modelling\>"

In [ ]:
# Changing the data types (to be more precise)
# Change to category
df_clean[['credit_mix', 'min_amount_payment', 'payment_behaviour', 'last_3_loan_type', 'last_2_loan_type', 'last_1_loan_type', 'credit_rating', 'gender', 'occupation']] = df_clean[['credit_mix', 'min_amount_payment', 'payment_behaviour',
                                                                                                                                                                  'last_3_loan_type', 'last_2_loan_type', 'last_1_loan_type', 'credit_rating',
                                                                                                                                                                    'gender', 'occupation']].astype('category')


# To float dtype
# The reason we have to do this because, these features are stored in months instead of years.
# And we have to transform it into year which might can be store in float data type (for example: 3 month might become 0.35 year).
df_clean['count_credit_history_years'] = df_clean['count_credit_history_years'].astype('float64')

In [ ]:
# Checking the result.
df_clean.dtypes.value_counts().to_frame('How many dtypes?')

In [ ]:
data_cleaning_1_explanations = """
To ensure that the algorithm will treat the variables correctly, correcting the data types helps to ensure that the feature will be analyzed (treated) correctly.
Furthermore, it help to creating encoding correctly.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### B.2 Fixing "\<Rename the column for convenience\>"

In [ ]:
# Rename _monthly_invested_amount
df_clean.rename(columns = {"_monthly_invested_amount" : "monthly_invested_amount"}, inplace = True)

# Raname count_credit_history_years to count_credit_history_month
# The reason is we have to do it beacuse we will create the new feature with year
df_clean.rename(columns = {"count_credit_history_years" : "count_credit_history_month"}, inplace = True)

In [ ]:
data_cleaning_2_explanations = """
Acually, this remaming a columns might not imporve the performamce but it help to reducing confusing when we rush or deal with overwhelming task.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### B.3 Fixing "\<Remove the incorrect values\>"

In [ ]:
## Removing some numeric columns which have incorrect values.
# min customer age = 14 is wrong which should be 18
df_clean['age'] = df_clean['age'].where(df_clean['age'] >= 18)

# min of count_bank_accounts = 0 must be inccorect value.
df_clean['count_bank_accounts'] = df_clean['count_bank_accounts'].where(df_clean['count_bank_accounts'] > 0)

# min of avg_days_past_due =  - 5  is weird (incorrect)
df_clean['avg_days_past_due'] = df_clean['avg_days_past_due'].where(df_clean['avg_days_past_due'] >= 0)

# min of monthly_emi_payment less than 0 is incorrect values.
df_clean['monthly_emi_payment'] = df_clean['monthly_emi_payment'].where(df_clean['monthly_emi_payment'] >= 0)

In [ ]:
## Dropping rows with null values
df_clean = df_clean.dropna().reset_index(drop=True)

In [ ]:
# Checking the result. (checking the shape after removing)
df_clean.info()

In [ ]:
# Checking the result. (after)
df_clean[['age', 'count_bank_accounts', 'avg_days_past_due', 'monthly_emi_payment']].describe().round(2)

In [ ]:
data_cleaning_3_explanations = """
The incorrect value is the most crucial part for modeling because it will mislead the model's performance, and when we rely on it, it will create a wrong strategy, leading to unexpected financial loss.
In this case, I decided to remove them because it did not result in data loss, and removing them is the best way to maximize the correctness of the dataset more than imputing values.
On the other hand, it make the result more precise and more directly related to the population.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

---
## C. Feature Engineering

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  df_eng = df_clean.copy()
except Exception as e:
  print(e)

### C.1 New Feature "\<credit_history_years\>"

Values in this feature are stored in the form of months, which is incorrect, and we have to transform them into years.

In [ ]:
# Creating the credit_history_years feature by transforming (divided by 12)
df_eng['credit_history_years'] = df_eng['count_credit_history_month']/12

# Droping the old one.
df_eng = df_eng.drop(columns = 'count_credit_history_month')

In [ ]:
# Checking the results
df_eng['credit_history_years'].describe().round(2)

In [ ]:
feature_engineering_1_explanations = """
This transformation helps in normalizing the values and correcting the values that ii should be kept (from month to year).
In practical, it might not help in improving classification model perfomace but I think this give a significant improvement to the model.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### C.2 New Feature "\<installment_ratio\>"



In [ ]:
# Creating the installment_ratio, which indicates the installment ratio that each customer has to pay.
df_eng['installment_ratio'] = df_eng['monthly_emi_payment']*12/df_eng['annual_income']

In [ ]:
feature_engineering_2_explanations = """
This feature measures the proportion of income that each customer have to keep for their scheduled repayments; higher values indicate tighter cash flow and elevated repayment risk.
From my opinion, this feature is directly related target variable which reflect customer the ratio that they have to pay based on their schedule.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### C.3 New Feature "\<debt_to_Income\>"



In [ ]:
# debt_to_Income feature
df_eng['debt_to_Income'] = df_eng['outstanding_debt']/df_eng['annual_income']

In [ ]:
feature_engineering_3_explanations = """
This feature indicates the earning power of each customer which higher value implies that they have a larger debt which might reduce their performance on their credit score.
From my perspective, the earning power of each customer reflect to credit_rating. It is mean, if they have more income (spending power), they will have ability to payback frequently and on time.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_3_explanations', value=feature_engineering_3_explanations)

### C.n Fixing "\<liquidity\>"

> You can add more cells related to new features in this section

In [ ]:
# Creating liquidity feature
df_eng['liquidity'] = df_eng['monthly_balance']*12/df_eng['annual_income']

In [ ]:
feature_engineering_n_explanations = """
This feature measures the monthly net balance / available balance, which indicates the liquidity buffer or the ability to take a risk when unexpected events happen (eg, loss in their business).
Putting this feature into the model will improve the model performance.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_n_explanations', value=feature_engineering_n_explanations)

---
## D. Data Preparation for Modeling

### D.1 Split Datasets

To minimise model overfitting, I decide to use stratified split to ensure the data of each group will be spltted almost equally wich help in reducing model overfitting (it ensure that each set have a similar propotion of every class).

On top of that, I will divide the data to 3 sets which has train (60%), val(20%), and testing (20%), because we want to ensure that the data that we have on validation and test sets has enough for adjusting the hyperparameter and generalise on the unseen data.

In [ ]:
# Splitting feature and target variable out before doing stratified split
y = df_eng['credit_rating']
X = df_eng.drop(columns=['credit_rating'])

In [ ]:
# Checking the frequency
y.value_counts(normalize=True)

In [ ]:
# Splitting test set for 20% of the dataset
from sklearn.model_selection import train_test_split
X_data, X_test, y_data, y_test = train_test_split(X, y, test_size = 0.20, stratify = y, random_state = 8)

In [ ]:
# Splitting 20% for validation set
X_train, X_val, y_train, y_val = train_test_split(X_data, y_data, test_size = 0.25, stratify = y_data, random_state = 8)

# In this case we use 0.25 because 25% of 80 is 20% in total data.

In [ ]:
# Checking the proportion of each set.
print(len(y_train), len(y_val), len(y_test))

In [ ]:
# Checking the group propotion of each set
print(y_train.value_counts(normalize=True))
print(y_val.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

# Each seem to have similar class's propotion

In [ ]:
# <Student to fill this section>
data_splitting_explanations = """
This splitting method will help to ensure that our splitting will all group of data equally that helping on reducing model's overfitting.
Due to classification models is very sensitive to the imbalance of the dataset and based on my knowledge, this is what I am capable of.
For the propotion of data splitting, To ensure that the model that we have will generalise well on unseen data. I believe 20% will be enough for generalising.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_splitting_explanations', value=data_splitting_explanations)

### A.2 Approach 2: ANOVA (after splitting)

Now, after we spliing the data (for prevenging data leakage), let find the feature that should be on our model.

In [ ]:
# Importing f_classif for calculating ANOVA for the classification model.
from sklearn.feature_selection import f_classif

# Selecting the numerical features
numerical_features = X_train.select_dtypes(include = 'number').columns

# apply ANOVA for each numerical feature.
f_stat, p_value = f_classif(X_train[numerical_features], y_train)   # According to the data have been cleaned. Directly apply ANOVA to the train set is considered no risk.

# Storing the result in form of columns
anova_result = (pd.DataFrame({"numerical features": numerical_features,
                              "F-statistic": f_stat,
                              "p-values": p_value}).sort_values("p-values"))

# Display the result.
anova_result.round(10)  ## From the result, every num feature are statistic significant toward the y_train althogh credit_ratio have the highest p-value at 0.007689

In [ ]:
# <Student to fill this section>
feature_selection_2_insights = """
ANOVA is a feature selection method for numerical columns toward the credit_rating which will evalute that the feature is statistic significant or not.
Instead of rely only on domain and business knowledge, using statistic testing help us to confirm what the capability of the features.
On top of that, these method is fast on calculating the result.
According to the result, every numerical columns that we have are statistically significant and we will put all of them into our model.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_2_insights', value=feature_selection_2_insights)

### A.3 Approach: Chi-squres (after splitting)


In [ ]:
from scipy.stats import chi2_contingency

# Then, put the categorical features in the list for
categorical_features = X_train.select_dtypes(include = 'category').columns

for col in categorical_features:   ## Using for loop to calculate all of the category features toward y_train
    chi = pd.crosstab(X_train[col], y_train)
    chi2, p, dof, expected = chi2_contingency(chi)
    print(f" \n{col}: Chi-square = {chi2:.2f}, p-value= {p:.10f}")

# 2 cat variables are not statistically significant toward the y_train which are gender and occupation.
# Then we will remove them from our model.

In [ ]:
# Remove gender and occupation out from train, validation and test set which is not statistically significant.
columns_to_drop = ['gender', 'occupation']

X_train = X_train.drop(columns = columns_to_drop)
X_val = X_val.drop(columns = columns_to_drop)
X_test = X_test.drop(columns = columns_to_drop)

In [ ]:
# <Student to fill this section>
feature_selection_n_insights = """
According to the result, gender and occupation are not statistically significant with p-values > 0.05 so we will remove them from our model.
credit_mix and min_amount_payment has the highest score indicating the huge different in distribution between each group.
We use this approach for tesing our categorical feature because it fast to test which features are significant or not.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_n_insights', value=feature_selection_n_insights)

### D.2 Data Transformation: One-hot encoding

We will trasform our potential categorical features to binary which the model can use to generalise.

* credit_mix
* min_amount_payment
* payment_behaviour
* last_1_loan_type
* last_2_loan_type
* last_3_loan_type

In [ ]:
# One hot encoding for categorical columns.
# Selecting the columns which have category dtype
cat_features = X_train.select_dtypes(include = 'category').columns.tolist()

# Creating dummy variable for each set
X_train = pd.get_dummies(X_train, columns = cat_features, drop_first=False) ## drop_first=False equal to include all dummy variables from the function.
X_val  = pd.get_dummies(X_val,  columns = cat_features, drop_first=False)
X_test = pd.get_dummies(X_test, columns = cat_features, drop_first=False)

# Then Reindex for vallidation and testing set to ensure that they are align with train set
X_val  = X_val.reindex(columns = X_train.columns, fill_value=0)
X_test = X_test.reindex(columns = X_train.columns, fill_value=0)

In [ ]:
# Checking the shape of each set.
print(X_train.shape)
print(" ")
print(X_val.shape)
print(" ")
print(X_test.shape)

# The data transformation have been done correctly.

In [ ]:
# <Student to fill this section and then remove this comment>
data_transformation_1_explanations = """
This transformation helps the model analyse on the categorical columns, as these features might directly influence the credit rating.
It will transform the categorical columns to binary columns, which the algorithm uses to learn these features.
I believe that these categorical feature will help the model to generalise well on the unseen data (the data which is not include in this dataset) which can be use in my advance model development in the future.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

---
## E. Save Datasets

> Do not change this code

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL

try:
  X_train.to_csv(at.folder_path / 'X_train.csv', index=False)
  y_train.to_csv(at.folder_path / 'y_train.csv', index=False)

  X_val.to_csv(at.folder_path / 'X_val.csv', index=False)
  y_val.to_csv(at.folder_path / 'y_val.csv', index=False)

  X_test.to_csv(at.folder_path / 'X_test.csv', index=False)
  y_test.to_csv(at.folder_path / 'y_test.csv', index=False)
except Exception as e:
  print(e)